# Projekt - fáza č. 2

In [5]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.stats.api as sms
import scipy.stats as stats
import os
from sklearn import preprocessing
from IPython.display import display
sns.set_theme(style="whitegrid")

## 2.1 A - Rozdelenie dát na trénovaciu a testovaciu množinu
Delíme pôvodný súbor `readings.csv` (bez čistenia) v pomere 80 % train a 20 % test. Každá jednotka (`id_unit`) má v readings viac meraní, v priemere asi 7, preto delíme podľa jednotiek, nie podľa riadkov. Všetky merania jednej jednotky skončia buď v train, alebo v test.

Delenie zároveň zachováva pomer tried Failure, takže podiel porúch je v oboch množinách približne rovnaký. `random_state` je pevne daný, aby bolo delenie vždy rovnaké.

Jediná úprava pred delením je prevod `id_unit` na číslo (`U00078` na 78) a oprava chybných ID s posunom 90 000, aby sa jednotky dali správne zoskupiť.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42
TEST_SIZE = 0.20

data = pd.read_csv("./data/readings.csv", sep="\t")

# id_unit potrebujeme ako číslo (U00078 na 78); chybné ID (> 2000) majú posun 90 000
data["id_unit"] = data["id_unit"].str[1:].astype(int)
data.loc[data["id_unit"] > 2000, "id_unit"] -= 90000

sgkf = StratifiedGroupKFold(n_splits=round(1 / TEST_SIZE), shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(sgkf.split(data, data["Failure"], groups=data["id_unit"]))
train = data.iloc[train_idx].copy()
test = data.iloc[test_idx].copy()

assert set(train["id_unit"]).isdisjoint(test["id_unit"])
assert len(train) + len(test) == len(data)

for nazov, cast in [("celé dáta", data), ("train", train), ("test", test)]:
    print(nazov, "- riadky:", len(cast), "- jednotky:", cast["id_unit"].nunique(),
          "- podiel Failure=1:", round(cast["Failure"].mean(), 4))

os.makedirs("./data/Ciste_data", exist_ok=True)
train.to_csv("./data/Ciste_data/train_raw.csv", index=False, encoding="utf-8-sig")
test.to_csv("./data/Ciste_data/test_raw.csv", index=False, encoding="utf-8-sig")

celé dáta - riadky: 14052 - jednotky: 1999 - podiel Failure=1: 0.1188
train - riadky: 11244 - jednotky: 1597 - podiel Failure=1: 0.1189
test - riadky: 2808 - jednotky: 402 - podiel Failure=1: 0.1186
